[![Open In Colab](/_static/colab-badge.svg)](https://colab.research.google.com/github/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)
[![Get Notebook](/_static/get-notebook-badge.svg)](https://raw.githubusercontent.com/OpenProteinAI/openprotein-docs/refs/heads/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)
[![View In GitHub](/_static/view-in-github-badge.svg)](https://github.com/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)

# Using ESMFold2

ESMFold2 is the latest generation of the ESM structure-prediction family
([Biohub/esm](https://github.com/Biohub/esm)). Unlike first-generation
ESMFold, it predicts full biomolecular *complexes* (multiple protein
chains, nucleic acids, and small-molecule ligands) with a
diffusion-based decoder, and can optionally condition on a multiple
sequence alignment (MSA) for improved accuracy.

Two variants are available:

- `` :py:class:`~openprotein.fold.ESMFold2Model` ``{=rst}
  (`session.fold.esmfold2`): the full model; accepts an optional MSA per
  protein chain.
- `` :py:class:`~openprotein.fold.ESMFold2FastModel` ``{=rst}
  (`session.fold.esmfold2_fast`): a single-sequence variant for fast
  predictions without an MSA. It uses half the folding layers of the
  full model.

Throughout this guide we use the mature 99-residue **HIV-1 protease**, a
classic obligate homodimer: two identical chains pair up to form the
functional enzyme, with a single active site at their interface. It is a
long-standing target of antiretroviral drugs such as ritonavir, which we
fold alongside it later.


## What you need before getting started

Make sure you have an active `OpenProtein` session, then import the
classes used to assemble a complex and define the protease sequence:


In [ ]:
import openprotein
from openprotein.molecules import Complex, Protein, Ligand

# Login to your session
session = openprotein.connect()

# Mature HIV-1 protease (99 residues), a classic obligate homodimer
sequence = (
    "PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGI"
    "GGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF"
)

## Getting the model

Create the model object and inspect its `fold` signature:


In [ ]:
esmfold2 = session.fold.esmfold2
help(esmfold2.fold)

## Folding a complex

The functional protease is a homodimer: two identical chains whose
catalytic site forms at their interface, so we fold both chains together
as a `` :py:class:`~openprotein.molecules.Complex` ``{=rst}. Because the
two chains are identical, we build a single
`` :py:class:`~openprotein.molecules.Protein` ``{=rst}, tag it with
`Protein.single_sequence_mode` to fold without an MSA, and place it in
both chain slots.

The runtime hyperparameters trade speed for accuracy:

- `num_recycles`: how many times the trunk refines its representation.
- `num_steps`: diffusion sampling steps. The default is 200, but fewer
  often suffice with little loss in quality; we use 50.
- `diffusion_samples`: number of independent structures drawn per input.
- `seed`: makes a run reproducible.

Submitting a fold returns a
`` :py:class:`~openprotein.fold.FoldResultFuture` ``{=rst}; wait for it
with `` :py:meth:`~openprotein.jobs.Future.wait_until_done` ``{=rst}:


In [ ]:
protease = Protein(sequence)
protease.msa = Protein.single_sequence_mode

dimer = Complex(chains={"A": protease, "B": protease})

dimer_future = esmfold2.fold(
    [dimer],
    num_recycles=3,        # trunk recycling iterations
    num_steps=50,          # diffusion sampling steps (default 200)
    diffusion_samples=1,   # structures drawn per input
    seed=0,
)
dimer_future.wait_until_done(verbose=True, timeout=900)

## Retrieving and visualizing the structure

### Getting the structure

Fetch results with
`` :py:meth:`~openprotein.fold.FoldResultFuture.get` ``{=rst}. It returns
one `` :py:class:`~openprotein.molecules.Structure` ``{=rst} per input,
and each `Structure` holds one
`` :py:class:`~openprotein.molecules.Complex` ``{=rst} per diffusion
sample:


In [ ]:
structure = dimer_future.get()[0]
predicted = structure[0]              # first diffusion sample

print("Predicted structure:", structure)
print("Chain A:", predicted.get_protein("A").sequence.decode())
print("Chain B:", predicted.get_protein("B").sequence.decode())

### Visualizing the structure

Render the prediction with
[molviewspec](https://github.com/molstar/mol-view-spec), coloring each
chain separately so the two halves of the dimer are easy to tell apart:


In [ ]:
%pip install molviewspec
from molviewspec import create_builder

def display_structure(structure_string):
    builder = create_builder()
    (
        builder.download(url="mystructure.cif")
        .parse(format="mmcif")
        .model_structure()
        .component()
        .representation()
        .color_from_source(
            schema="atom",
            category_name="atom_site",
            field_name="auth_asym_id",
            palette={
                "kind": "categorical",          # color by chain
                "colors": ["blue", "red", "green", "orange"],
                "mode": "ordinal",
            },
        )
    )
    return builder.molstar_notebook(
        data={"mystructure.cif": structure_string}, width=500, height=400
    )

display_structure(structure.to_string(format="cif"))

## Assessing prediction confidence

ESMFold2 reports per-sample confidence via
`` :py:meth:`~openprotein.fold.FoldResultFuture.get_confidence` ``{=rst},
returning one
`` :py:class:`~openprotein.fold.ESMFold2Confidence` ``{=rst} per
diffusion sample. For a multi-chain complex the scores describe both the
whole assembly and the relationships between chains:

- **pTM** (predicted TM-score): global accuracy of the overall fold
  (0–1, higher is better).
- **ipTM** (interface pTM): accuracy of the *relative placement* of the
  chains; the key score for whether the interface is right.
- **complex pLDDT**: mean per-residue confidence across the complex.
- **per-chain pTM** (`chains_ptm`): pTM computed for each chain alone.
- **pairwise chain ipTM** (`pair_chains_iptm`): ipTM for each ordered
  pair of chains, i.e. how confidently chain *i* is placed relative to
  chain *j*.

A single chain has no interface, so ipTM and the pairwise table only
become meaningful once you fold two or more chains, as we do here:


In [ ]:
confidence = dimer_future.get_confidence()[0][0]   # [input][diffusion sample]

print(f"pTM:           {confidence.ptm:.3f}")
print(f"ipTM:          {confidence.iptm:.3f}")
print(f"complex pLDDT: {confidence.complex_plddt:.3f}")
print("per-chain pTM: " + ", ".join(
    f"{cid}={val:.3f}" for cid, val in confidence.chains_ptm.items()
))

# Pairwise chain ipTM, as a labeled grid (row placed relative to column)
pair = confidence.pair_chains_iptm
chain_ids = list(pair.keys())
print("\npairwise chain ipTM:")
print("       " + "".join(f"{c:>8}" for c in chain_ids))
for i in chain_ids:
    row = "".join(f"{pair[i][j]:8.3f}" for j in chain_ids)
    print(f"{i:>5}  {row}")

### Visualizing PAE and pLDDT

`` :py:meth:`~openprotein.fold.FoldResultFuture.get_pae` ``{=rst} returns
the Predicted Aligned Error: an N×N matrix (in ångströms) whose entry
(i, j) is the expected error in residue *j*'s position when the
structure is aligned on residue *i*. Confident off-diagonal blocks
between the two chains signal a well-predicted interface.
`` :py:meth:`~openprotein.fold.FoldResultFuture.get_plddt` ``{=rst}
returns the per-residue pLDDT. Both arrive with a leading
diffusion-sample axis, so we take the first sample:


In [ ]:
import matplotlib.pyplot as plt

pae = dimer_future.get_pae()[0][0]      # (samples, N, N) -> first sample
plddt = dimer_future.get_plddt()[0][0]  # (samples, N)    -> first sample

fig, (ax_pae, ax_plddt) = plt.subplots(1, 2, figsize=(11, 4))

im = ax_pae.imshow(pae, cmap="Greens_r", vmin=0)
ax_pae.set(title="Predicted Aligned Error", xlabel="residue", ylabel="residue")
fig.colorbar(im, ax=ax_pae, label="expected error (Å)")

ax_plddt.plot(plddt)
ax_plddt.set(title="pLDDT", xlabel="residue", ylabel="confidence", ylim=(0, 1))
ax_plddt.axvline(len(plddt) // 2, ls="--", color="grey")  # chain A / B boundary

plt.tight_layout()
plt.show()

## Conditioning on an MSA

The folds so far used single-sequence mode, which already gives ESMFold2
strong accuracy. Conditioning each protein chain on a multiple sequence
alignment can improve it further; the full `esmfold2` model supports
this as an option. Build an MSA from the query sequence with
`` :py:meth:`session.align.create_msa <openprotein.align.AlignAPI.create_msa>` ``{=rst}
and assign it to the chain's `msa` attribute. Both chains are identical,
so we attach the MSA to a single `Protein` and place it in both chain
slots:


In [ ]:
msa = session.align.create_msa(sequence.encode())

msa_protease = Protein(sequence)
msa_protease.msa = msa

msa_dimer = Complex(chains={"A": msa_protease, "B": msa_protease})

msa_future = esmfold2.fold([msa_dimer], num_steps=50, seed=0)
msa_future.wait_until_done(verbose=True, timeout=900)

## Folding with ligands, DNA, and RNA

Because ESMFold2 predicts whole complexes, a `Complex` can mix protein
chains with small-molecule **ligands**, **DNA**, and **RNA**. Here we
co-fold the protease dimer with ritonavir, an antiretroviral drug
designed to inhibit it, supplied as a
`` :py:class:`~openprotein.molecules.Ligand` ``{=rst} by its Chemical
Component Dictionary (CCD) code:


In [ ]:
ligand_dimer = Complex(chains={
    "A": protease,
    "B": protease,
    "L": Ligand(ccd="RIT"),   # ritonavir, by CCD code
    # ...a ligand can also be given a SMILES string: Ligand(smiles=...)
})

ligand_future = esmfold2.fold([ligand_dimer], num_steps=50, seed=0)
ligand_future.wait_until_done(verbose=True, timeout=900)

DNA and RNA chains follow the same pattern: build them from a
nucleotide sequence and add them under their own chain id. The protease
does not bind nucleic acids, but the API looks like this:

```python
from openprotein.molecules import DNA, RNA

nucleic_complex = Complex(chains={
    "A": protease,
    "D": DNA("GGAATTCC"),   # a DNA chain
    "R": RNA("GGGAGG"),     # an RNA chain
})
```


## Using ESMFold2-Fast

`esmfold2_fast` is a lighter-weight, faster variant that uses half the
folding layers, making it well-suited to high-throughput screens where
you need to fold many sequences quickly. It takes the same inputs and
hyperparameters, but every protein chain must use
`Protein.single_sequence_mode`; it rejects chains that carry an MSA:


In [ ]:
esmfold2_fast = session.fold.esmfold2_fast

fast_chain = Protein(sequence)
fast_chain.msa = Protein.single_sequence_mode

fast_future = esmfold2_fast.fold([fast_chain], num_steps=50, seed=0)
fast_future.wait_until_done(verbose=True, timeout=900)

## A note on ESMFold (first generation)

First-generation **ESMFold** remains available via `session.fold.esmfold`
for quick single-sequence protein predictions. It does *not* support
ligands, nucleic acids, or MSA conditioning, but it is a fast option when
you only need to fold protein chains:


In [ ]:
esmfold = session.fold.esmfold
esm_structure = esmfold.fold([sequence.encode()]).get()[0]
print("ESMFold (v1) prediction:", esm_structure)

## Next steps

Save a prediction for later, or compare ESMFold2 against another
predictor such as [AlphaFold2](./Using_AlphaFold2.ipynb),
[Boltz](./Using_Boltz_1_and_Boltz_2.ipynb), or
[Protenix](./Using_Protenix.ipynb):


In [ ]:
with open("esmfold2_prediction.cif", "w") as f:
    f.write(structure.to_string(format="cif"))